# Generative Models Demonstration In Python

## Background 

Credit scoring is a critical function in retail banking and consumer finance, enabling financial institutions to evaluate the creditworthiness of individuals before granting loans, credit cards, or overdraft facilities. This assessment relies heavily on historical customer data, including demographic details, financial behavior, and repayment patterns.

## Objective
To generate realistic synthetic credit data using GANs that preserves the statistical patterns and relationships of the original dataset.

## What is GAN
- Generative models are designed to learn the underlying probability distribution of a dataset, enabling them to generate new data that preserves univariate properties, multivariate relationships, and correlation structures observed in the original data.

- In other words, Generative models learn the data distribution and generate new samples that retain statistical properties such as distributions and correlations.

## What is CT-GAN
- CTGAN (Conditional Tabular GAN) is a generative model specifically designed for tabular data, capable of handling both continuous and categorical variables effectively.

- It uses conditional generation to model complex relationships and address issues like imbalanced categories by sampling data conditionally.

### Import Libraries

In [1]:
from ctgan import CTGAN
import pandas as pd

### Load Data

In [2]:
data = pd.read_csv("BANK LOAN.csv")
data.drop(columns = [ 'SN'],axis = 1, inplace = True)
data

,AGE,EMPLOY,ADDRESS,DEBTINC,CREDDEBT,OTHDEBT,DEFAULTER
0,3,17,12,9.3,11.36,5.01,1
1,1,10,6,17.3,1.36,4.00,0
2,2,15,14,5.5,0.86,2.17,0
3,3,15,14,2.9,2.66,0.82,0
4,1,2,0,17.3,1.79,3.06,1
...,...,...,...,...,...,...,...
695,2,6,15,4.6,0.26,0.98,1
696,1,6,4,11.5,0.37,2.05,0
697,2,15,3,7.6,0.49,1.94,0
698,3,19,22,8.4,2.30,4.17,0


### Training the CTGAN Model

This step initializes the CTGAN model with 200 epochs and trains it on the dataset to learn its underlying distribution for synthetic data generation.

In [15]:
ctgan = CTGAN(epochs=200)
ctgan.fit(data, discrete_columns=['AGE', 'DEFAULTER','EMPLOY','ADDRESS'])

### Generate synthetic data


In [16]:
synthetic_data = ctgan.sample(5000)
synthetic_data

,AGE,EMPLOY,ADDRESS,DEBTINC,CREDDEBT,OTHDEBT,DEFAULTER
0,2,1,4,12.140274,1.067073,-1.009212,0
1,3,13,16,31.864389,0.845551,6.055542,0
2,3,1,7,9.626321,1.330613,-0.932139,0
3,3,17,2,13.454380,1.218001,-5.640557,0
4,2,4,22,30.524817,1.217965,-0.310117,0
...,...,...,...,...,...,...,...
4995,2,12,4,17.818719,1.110491,-0.848303,0
4996,2,2,9,30.247045,0.984174,2.423878,1
4997,3,16,0,6.552887,2.446862,-0.459290,1
4998,3,19,10,8.095996,0.720044,-0.202478,0


### Statistical Comparison of Original vs Synthetic Data

**Note:**


Since Age represents categorical groups (despite being numerically encoded), mean and standard deviation are not meaningful measures. Therefore, we evaluate it using frequency and percentage distribution instead.

In [21]:
# ---- Numeric comparison ----
num_cols = data.select_dtypes(include='number').columns.drop('AGE')

numeric_comparison = pd.concat([
    data[num_cols].agg(['mean','std']).T.add_prefix('Orig_'),
    synthetic_data[num_cols].agg(['mean','std']).T.add_prefix('Syn_')
], axis=1)

# ---- AGE (categorical) comparison ----
age_comparison = pd.concat([
    data['AGE'].value_counts().sort_index().rename('Orig_Freq'),
    (data['AGE'].value_counts(normalize=True).sort_index()*100).rename('Orig_%'),
    synthetic_data['AGE'].value_counts().sort_index().rename('Syn_Freq'),
    (synthetic_data['AGE'].value_counts(normalize=True).sort_index()*100).rename('Syn_%')
], axis=1).round(2)

# Display
print("Numeric Comparison:\n", numeric_comparison.round(2))
print("\nAGE Distribution:\n", age_comparison.round(2))


Numeric Comparison:
            Orig_mean  Orig_std  Syn_mean  Syn_std
EMPLOY          8.39      6.66      9.70     7.60
ADDRESS         8.28      6.82      9.40     7.54
DEBTINC        10.26      6.83     16.18    10.24
CREDDEBT        1.55      2.12      2.27     2.52
OTHDEBT         3.06      3.29      1.06     2.88
DEFAULTER       0.26      0.44      0.27     0.45

AGE Distribution:
      Orig_Freq  Orig_%  Syn_Freq  Syn_%
AGE                                    
1          242   34.57      1469  29.38
2          284   40.57      1998  39.96
3          174   24.86      1533  30.66
